# RideTime Direct Lake reporting tables
Run after ThemePark_WaitTime_Collector and before refreshing the Direct Lake
semantic model. Rebuilds three small demo tables; never changes the raw tables.
Full-history replacement is intentional for this small demo, not a large-scale
incremental processing pattern.

In [ ]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")
fact = spark.table("dbo.fact_wait_snapshot").cache()
try:
    row_count = fact.count()
    if row_count == 0:
        raise ValueError("No snapshots collected; run the collector first.")
    if fact.filter("snapshot_utc IS NULL OR entity_id IS NULL").limit(1).count():
        raise ValueError("Snapshot timestamp and entity key must not be null.")
    if fact.groupBy("snapshot_utc", "entity_id").count().filter("count > 1").limit(1).count():
        raise ValueError("Duplicate attraction/collection keys in the source.")
    entities = spark.table("dbo.dim_entity")
    if entities.filter("entity_id IS NULL").limit(1).count():
        raise ValueError("Entity dimension contains a null key.")
    if entities.groupBy("entity_id").count().filter("count > 1").limit(1).count():
        raise ValueError("Entity dimension contains duplicate keys.")
    if fact.join(entities, "entity_id", "left_anti").limit(1).count():
        raise ValueError("Snapshots contain entity keys absent from the dimension.")

    bounds = fact.agg(F.min("snapshot_utc"), F.max("snapshot_utc")).first()
    first_year, last_year = bounds[0].year, bounds[1].year
    dates = spark.sql(f"""
        SELECT explode(sequence(
            DATE '{first_year}-01-01', DATE '{last_year}-12-31',
            INTERVAL 1 DAY)) AS date
    """).selectExpr(
        "date",
        "year(date) AS year",
        "concat('Q', quarter(date)) AS quarter",
        "month(date) AS month_number",
        "date_format(date, 'MMM') AS month",
        "date_format(date, 'yyyy-MM') AS year_month",
        "year(date) * 100 + month(date) AS year_month_sort",
        "pmod(dayofweek(date) + 5, 7) + 1 AS day_of_week_number",
        "date_format(date, 'EEE') AS day_of_week",
        "dayofweek(date) IN (1, 7) AS is_weekend",
    )
    hours = spark.range(24).selectExpr(
        "CAST(id AS INT) AS hour",
        "concat(lpad(CAST(id AS STRING), 2, '0'), ':00') AS hour_label",
    )
    reporting = fact.withColumn(
        "snapshot_date_utc", F.to_date("snapshot_utc")
    ).withColumn("snapshot_hour_utc", F.hour("snapshot_utc"))

    # Publish dimensions first so newly added dates exist before the fact.
    for name, data in (
        ("dim_date_utc", dates),
        ("dim_hour_utc", hours),
        ("fact_wait_snapshot_reporting", reporting),
    ):
        data.write.format("delta").mode("overwrite").option(
            "overwriteSchema", "true"
        ).saveAsTable(f"dbo.{name}")

    saved = spark.table("dbo.fact_wait_snapshot_reporting")
    if saved.count() != row_count:
        raise ValueError("Reporting row count does not match the captured source.")
    if saved.join(dates, saved.snapshot_date_utc == dates.date, "left_anti").limit(1).count():
        raise ValueError("Reporting fact has unmatched date keys.")
    if saved.join(hours, saved.snapshot_hour_utc == hours.hour, "left_anti").limit(1).count():
        raise ValueError("Reporting fact has unmatched hour keys.")
    print(f"Validated {row_count:,} snapshots, {dates.count()} dates, {hours.count()} hours.")
    print(f"Latest collection UTC: {bounds[1]}")
finally:
    fact.unpersist()